# **Illustrated Word2Vec**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Introduction](#Introduction)
- [Personality Embeddings: What are you like?](#Personality-Embeddings:-What-are-you-like?)
- [Word Embeddings](#Word-Embeddings)
  - [Analogies](#Analogies)
- [Language Modeling](#Language-Modeling)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Introduction

The concept of embeddings is one of the most fascinating ideas in machine learning. If you’ve ever used Siri, Google Assistant, Alexa, Google Translate, or even smartphone keyboard with next-word prediction, then chances are you’ve benefitted from this idea that has become central to Natural Language Processing (NLP) models. There has been quite a development over the last couple of decades in using embeddings for neural models.

Word2Vec is a method to efficiently create word embeddings and has been around since 2013. But in addition to its utility as a word-embedding method, some of its concepts have been shown to be effective in creating recommendation engines and making sense of sequential data even in commercial, non-language tasks. Companies like Airbnb, Alibaba, Spotify, and Anghami have all benefitted from carving out this brilliant piece of machinery from the world of NLP and using it in production to empower a new breed of recommendation engines.

In this notebook, we’ll go over the concept of embedding, and the mechanics of generating embeddings with Word2Vec. But let’s start with an example to get familiar with using vectors to represent things.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Personality Embeddings: What are you like?

On a scale of 0 to 100, how introverted/extroverted are you (where 0 is the most introverted, and 100 is the most extroverted)? Have you ever taken a personality test like MBTI – or even better, the Big Five Personality Traits test? If you haven’t, these are tests that ask you a list of questions, then score you on a number of axes, introversion/extroversion being one of them.

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Example of the result of a Big Five Personality Trait test.<br> It can really tell you a lot about yourself and is shown to have<br> predictive ability in academic, personal, and professional success.</em>
    </td>
  </tr>
</table>

Imagine I’ve scored 38/100 as my introversion/extroversion score. we can plot that in this way:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-02.png" width="400">

Let’s switch the range to be from -1 to 1:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-03.png" width="400">

How well do you feel you know a person knowing only this one piece of information about them? Not much. People are complex. So let’s add another dimension – the score of one other trait from the test.

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-04.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>We can represent the two dimensions as a point on the graph, or better yet,<br> as a vector from the origin to that point. We have incredible tools to deal with<br> vectors that will come in handy very shortly.</em>
    </td>
  </tr>
</table>

I’ve hidden which traits we’re plotting just so you get used to not knowing what each dimension represents – but still getting a lot of value from the vector representation of a person’s personality.

We can now say that this vector partially represents my personality. The usefulness of such representation comes when you want to compare two other people to me. Say I get hit by a $\color{red}{\text{bus}}$ and I need to be replaced by someone with a similar personality. In the following figure, which of the two people is more similar to me?

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-05.png" width="500">

When dealing with vectors, a common way to calculate a similarity score is cosine similarity:

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-06.png" width="500">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Person #1 is more similar to me in personality.<br> Vectors pointing at the same direction (length plays a role as well)<br> have a higher cosine similarity score.</em>
    </td>
  </tr>
</table>

Yet again, two dimensions aren’t enough to capture enough information about how different people are. Decades of psychology research have led to five major traits (and plenty of sub-traits). So let’s use all five dimensions in our comparison:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-07.png" width="400">

The problem with five dimensions is that we lose the ability to draw neat little arrows in two dimensions. This is a common challenge in machine learning where we often have to think in higher-dimensional space. The good thing is, though, that cosine_similarity still works. It works with any number of dimensions:

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-08.png" width="500">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>cosine_similarity works for any number of dimensions.<br> These are much better scores because they're calculated based on<br> a higher resolution representation of the things being compared.</em>
    </td>
  </tr>
</table>

At the end of this section, I want us to come out with two central ideas:

1. We can represent people (and things) as vectors of numbers (which is great for machines!).
2. We can easily calculate how similar vectors are to each other.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-09.png" width="500">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Word Embeddings

With this understanding, we can proceed to look at trained word-vector examples (also called word embeddings) and start looking at some of their interesting properties.

This is a word embedding for the word “king” (GloVe vector trained on Wikipedia):


<div style="word-wrap: break-word; white-space: normal; line-height: 2.0; font-family: monospace; color: red;">
[ 0.50451, 0.68607, -0.59517, -0.022801, 0.60046, -0.13498, -0.08813, 0.47377, -0.61798, -0.31012, -0.076666, 1.493, -0.034189, -0.98173, 0.68229, 0.81722, -0.51874, -0.31503, -0.55809, 0.66421, 0.1961, -0.13495, -0.11476, -0.30344, 0.41177, -2.223, -1.0756, -1.0783, -0.34354, 0.33505, 1.9927, -0.04234, -0.64319, 0.71125, 0.49159, 0.16754, 0.34344, -0.25663, -0.8523, 0.1661, 0.40102, 1.1685, -1.0137, -0.21585, -0.15155, 0.78321, -0.91241, -1.6106, -0.64426, -0.51042 ]
</div>

It’s a list of 50 numbers. We can’t tell much by looking at the values. But let’s visualize it a bit so we can compare it with other word vectors. Let’s put all these numbers in one row:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-10.png">

Let’s color code the cells based on their values (red if they’re close to 2, white if they’re close to 0, blue if they’re close to -2):

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-11.png">

We’ll proceed by ignoring the numbers and only looking at the colors to indicate the values of the cells. Let’s now contrast “King” against other words:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-12.png">

See how “Man” and “Woman” are much more similar to each other than either of them is to “king”? This tells you something. These vector representations capture quite a bit of the information/meaning/associations of these words.

Here’s another list of examples (compare by vertically scanning the columns looking for columns with similar colors):

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-13.png">

A few things to point out:

1. There’s a straight red column through all of these different words. They’re similar along that dimension (and we don’t know what each dimensions codes for)
2. You can see how “woman” and “girl” are similar to each other in a lot of places. The same with “man” and “boy”
3. “boy” and “girl” also have places where they are similar to each other, but different from “woman” or “man”. Could these be coding for a vague conception of youth? possible.
4. All but the last word are words representing people. I added an object (water) to show the differences between categories. You can, for example, see that blue column going all the way down and stopping before the embedding for “water”.
5. There are clear places where “king” and “queen” are similar to each other and distinct from all the others. Could these be coding for a vague concept of royalty?

## Analogies

The famous examples that show an incredible property of embeddings is the concept of analogies. We can add and subtract word embeddings and arrive at interesting results. The most famous example is the formula: “king” - “man” + “woman”:

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-14.png">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Using the <a href="https://radimrehurek.com/gensim/" target="_blank">Gensim</a> library in python, we can add and subtract word vectors,<br> and it would find the most similar words to the resulting vector.<br> The image shows a list of the most similar words, each with its cosine similarity.</em>
    </td>
  </tr>
</table>

We can visualize this analogy as we did previously:

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-15.png">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>The resulting vector from "king-man+woman" doesn't exactly equal "queen",<br> but "queen" is the closest word to it from the 400,000 word embeddings we have in this collection.</em>
    </td>
  </tr>
</table>

Now that we’ve looked at trained word embeddings, let’s learn more about the training process. But before we get to Word2Vec, we need to look at a conceptual parent of word embeddings: the neural language model.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Language Modeling

If one wanted to give an example of an NLP application, one of the best examples would be the next-word prediction feature of a smartphone keyboard. It’s a feature that billions of people use hundreds of times every day.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-16.png">

Next-word prediction is a task that can be addressed by a *language model*. A language model can take a list of words (let’s say two words), and attempt to predict the word that follows them.

In the screenshot above, we can think of the model as one that took in these two green words ($\color{green}{\text{thou shalt}}$) and returned a list of suggestions (“not” being the one with the highest probability):

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-17.png" width="400">

We can think of the model as looking like this black box:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-18.png" width="500">

But in practice, the model doesn’t output only one word. It actually outputs a probability score for all the words it knows (the model’s “vocabulary”, which can range from a few thousand to over a million words). The keyboard application then has to find the words with the highest scores, and present those to the user.

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-19.png" width="600">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>The output of the neural language model is a probability score for all the words the model knows.<br> We're referring to the probability as a percentage here,<br> but 40% would actually be represented as 0.4 in the output vector.</em>
    </td>
  </tr>
</table>

After being trained, early neural language models ([A Neural Probabilistic Language Model, Bengio 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)) would calculate a prediction in three steps:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-20.png" width="600">

The first step is the most relevant for us as we discuss embeddings. One of the results of the training process was this matrix that contains an embedding for each word in our vocabulary. During prediction time, we just look up the embeddings of the input word, and use them to calculate the prediction:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Illustrated_Word2Vec-21.png" width="600">

Let’s now turn to the training process to learn more about how this embedding matrix was developed.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[The Illustrated Word2vec by Jay Alammar](https://jalammar.github.io/illustrated-word2vec/)**
- **Key Papers:**
    - [Word2Vec Paper, 2013](https://arxiv.org/pdf/1301.3781)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)